# 3.ControlNet with Canny Edge Detection

This script implements the third step of the system and can be executed separately to test generation performance.

Since this system is not publicly deployed and is built solely for academic assignment demonstration, the safety checker is disabled. This does not violate the terms of the Stable Diffusion license.

In [1]:
import os
import torch
import cv2
import numpy as np
from PIL import Image, ImageEnhance
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel
import ipywidgets as widgets
from IPython.display import display, clear_output

# ================= 1. Load Local Models (SD 1.5 + ControlNet Canny) =================
model_dir = "./stable-diffusion-v1-5"
controlnet_dir = "./controlnet-canny"

if not os.path.exists(controlnet_dir):
    raise FileNotFoundError("Please ensure the folder './controlnet-canny' and its weights exist locally.")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading ControlNet edge-locking controller...")
controlnet = ControlNetModel.from_pretrained(controlnet_dir, torch_dtype=torch_dtype)

print("Loading SD 1.5 Img2Img ControlNet pipeline...")
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    model_dir,
    controlnet=controlnet,
    torch_dtype=torch_dtype,
    safety_checker=None
).to(device)

if device == "cuda":
    pipe.enable_attention_slicing()
print("[Success] Adaptive high-contrast control pipeline loaded completely!")


# ================= 2. Build Interactive UI Widgets =================
img_path_input = widgets.Text(
    value='input.png',
    placeholder='Enter source image file path',
    description='Source Image Path:'
)

# Style Preservation Slider
style_preservation_slider = widgets.FloatSlider(
    value=0.75, min=0.10, max=0.90, step=0.05,
    description='Style Preservation:',
    tooltip='Base style retention strength. Note: If ink density or brush stroke is set to extreme values, this parameter will auto-adjust to boost detail contrast.'
)

# Negative Space Ratio Slider
negative_space_slider = widgets.IntSlider(
    value=50, min=0, max=100, step=5,
    description='Negative Space Ratio:'
)

# Ink Depth Slider (0 = Faint Light Ink, 100 = Dense Dark Dry Ink)
ink_depth_slider = widgets.IntSlider(
    value=50, min=0, max=100, step=5,
    description='Ink Density:'
)

# Brush Wetness Slider (0 = Wet Splash Ink, 100 = Dry Scratch Flying White Strokes)
brush_slider = widgets.IntSlider(
    value=50, min=0, max=100, step=5,
    description='Brush Wetness:'
)

btn_generate = widgets.Button(
    description='Start Precise Redraw (Physical Preprocessing Version)',
    button_style='warning',
    icon='paint-brush'
)

output_area = widgets.Output()

# ================= 3. Core Generation Logic (Aggressive Physical Preprocessing + Dynamic Parameter Linkage) =================
def on_button_clicked(b):
    with output_area:
        clear_output()
        img_path = img_path_input.value
        
        if not os.path.exists(img_path):
            print(f"❌ Source image not found: {img_path}")
            return
            
        # ---------- 3.1 Read Slider Values ----------
        ink_depth = ink_depth_slider.value          # Range: 0-100
        brush_val = brush_slider.value              # Range: 0-100
        neg_space_val = negative_space_slider.value # Range: 0-100

        # ---------- 3.2 Load Source Image & Convert to Numpy Array ----------
        init_image = Image.open(img_path).convert("RGB").resize((512, 512))
        image_np = np.array(init_image)

        print("⚡ Executing physical preprocessing to enhance visual effects...")

        # ---------- 3.3 [Ink Density] Physically Adjust Image Brightness & Contrast ----------
        # Map ink depth to brightness coefficient (0.3 ~ 1.8): Lower value = darker ink, higher value = lighter faded ink
        brightness_factor = 1.8 - (ink_depth / 100) * 1.5  
        contrast_factor = 0.6 + (ink_depth / 100) * 1.4    
        
        enhancer_b = ImageEnhance.Brightness(init_image)
        init_image = enhancer_b.enhance(brightness_factor)
        enhancer_c = ImageEnhance.Contrast(init_image)
        init_image = enhancer_c.enhance(contrast_factor)
        
        # Extra faint ink (value <20): Overlay gray mask to simulate water-washed faded effect
        if ink_depth < 20:
            grey_layer = Image.new('RGB', init_image.size, (180, 180, 180))
            init_image = Image.blend(init_image, grey_layer, 0.4)

        # ---------- 3.4 [Negative Space] Physically Adjust Color Saturation (Higher blank ratio = paler tones) ----------
        saturation_factor = 1.2 - (neg_space_val / 100) * 1.1  # Range: 1.2 ~ 0.1
        enhancer_s = ImageEnhance.Color(init_image)
        init_image = enhancer_s.enhance(max(0.1, saturation_factor))

        # ---------- 3.5 [Brush Wetness] Physically Modify Canny Edge Map (Key Edge Control) ----------
        # Extract raw edge map (fixed thresholds; lower values like 50,150 for more sensitive edges)
        canny_edges = cv2.Canny(image_np, 100, 200)
        
        if brush_val >= 70:  
            # [Extra Dry / Cracked Brush ]: Thin edges + random salt-pepper noise for grit texture
            kernel = np.ones((1, 1), np.uint8)
            canny_edges = cv2.erode(canny_edges, kernel, iterations=1)
            # Generate random white noise particles for flying white texture
            noise = np.random.randint(0, 255, canny_edges.shape, dtype=np.uint8)
            noise_mask = (noise > 230)
            canny_edges[noise_mask] = 255
            
        elif brush_val <= 30:  
            # [Extra Wet / Splash Ink ]: Dilate thick edges + heavy Gaussian blur for ink spreading
            kernel = np.ones((7, 7), np.uint8)
            canny_edges = cv2.dilate(canny_edges, kernel, iterations=2)
            canny_edges = cv2.GaussianBlur(canny_edges, (15, 15), 0)
            
        else:  
            # [Neutral Brush]: Mild blur for natural rice paper ink diffusion
            canny_edges = cv2.GaussianBlur(canny_edges, (3, 3), 0)
            
        # Convert single-channel edge map to 3-channel RGB (standard ControlNet input format)
        canny_edges = np.concatenate([canny_edges[:, :, None]] * 3, axis=2)
        canny_image = Image.fromarray(canny_edges)
        
        # Preview preprocessed edge map for user reference
        print("✅ Physical preprocessing complete! Preview the edge skeleton fed to ControlNet (modified per brush settings)")
        display(canny_image.resize((256, 256)))

        # ---------- 3.6 Dynamic Parameter Adaptation (Align Generation Engine with Preprocessing) ----------
        # A. Link brush wetness to ControlNet weight: Dry strokes lock edges (1.2), wet strokes relax edge constraint (0.6)
        if brush_val >= 70:
            ctrl_scale = 1.2
        elif brush_val <= 30:
            ctrl_scale = 0.6
        else:
            ctrl_scale = 0.9
            
        # B. Link negative space to CFG guidance scale: More blank space = higher CFG to enforce minimalist composition
        guidance = 7.0 + (neg_space_val / 100) * 4.0  # Range: 7.0 ~ 11.0
        
        # C. Style preservation (Denoising Strength) extreme adjustment:
        # Force heavy redraw (0.85) when ink/brush values hit extreme ranges for dramatic visual changes
        is_extreme = (ink_depth < 15 or ink_depth > 85 or brush_val < 15 or brush_val > 85)
        if is_extreme:
            final_strength = 0.85
            print(f"⚠️ Extreme parameter detected (ultra light/dark / dry/wet brush). Auto boost redraw strength to {final_strength}")
        else:
            final_strength = round(1.0 - style_preservation_slider.value, 2)

        # ---------- 3.7 Concise Powerful Prompts (Only for mood complement, physical preprocessing handles core visuals) ----------
        # Ink tone prompt fragments
        if ink_depth >= 80:
            ink_prompt = "extremely heavy dense pitch-black ink, high contrast"
        elif ink_depth <= 20:
            ink_prompt = "extremely pale water-diluted grey mist, very low contrast"
        else:
            ink_prompt = "balanced classic ink tones"

        # Brush stroke prompt fragments
        if brush_val >= 80:
            brush_prompt = "dry cracked flying-white scratches, sharp calligraphy lines"
        elif brush_val <= 20:
            brush_prompt = "heavy wet splash ink, blooming water bleeding diffusion effects"
        else:
            brush_prompt = "expressive traditional Chinese brushwork"

        # Negative space prompt fragments
        if neg_space_val >= 70:
            space_prompt = "huge minimalist negative space, vast empty blank rice paper"
        elif neg_space_val <= 30:
            space_prompt = "dense intricate ink texture filling entire canvas"
        else:
            space_prompt = "natural balanced composition with moderate blank space"

        prompt = f"masterpiece traditional chinese ink wash painting, {ink_prompt}, {brush_prompt}, {space_prompt}, monochrome ink wash on rice paper"
        negative_prompt = "colorful, 3d render, photorealistic, oil painting, modern art, messy clutter, frame, border, cartoon"

        print(f"🎯 Final Generation Params: Denoising Strength={final_strength}, ControlNet Scale={ctrl_scale}, CFG Scale={guidance}")
        print(f"📝 Prompt Fragments: {ink_prompt} | {brush_prompt} | {space_prompt}")

        # ---------- 3.8 Run AI Inference ----------
        try:
            with torch.inference_mode():
                result = pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    image=init_image,                    # Physically brightened/darkened/faded source image
                    control_image=canny_image,           # Blurred/eroded/noised edge map for edge locking
                    strength=final_strength,
                    controlnet_conditioning_scale=ctrl_scale,
                    guidance_scale=guidance,
                    num_inference_steps=35
                ).images[0]
                
            # ---------- 3.9 Post-processing: Overlay rice paper texture for minimalist blank space ----------
            if neg_space_val > 80:
                texture_path = "./rice_paper.jpg"
                if os.path.exists(texture_path):
                    texture = Image.open(texture_path).convert("L").resize(result.size)
                    texture_rgb = texture.convert("RGB")
                    result = Image.blend(result, texture_rgb, 0.2)
                    print("🖼️ Rice paper texture overlay applied to enhance vintage blank-space texture.")
                else:
                    print("ℹ️ rice_paper.jpg not found, skip texture overlay.")
                    
            # Preview and save output
            display(result)
            result.save("controlnet_contrast_output.jpg")
            print("✅ Generation completed successfully! Output saved locally as controlnet_contrast_output.jpg")
            
        except Exception as e:
            print(f"❌ Generation failed: {e}")

# ================= 4. Bind Button Trigger & Assemble Full UI Layouts =================
btn_generate.on_click(on_button_clicked)

ui = widgets.VBox([
    widgets.HTML(value="<h3>🖌️ Precise Ink Edge Lock Editor (Aggressive Physical Preprocessing Version)</h3>"),
    widgets.HTML(value="<i>Drag sliders for dramatic visual adjustments: dark dense ink, light faded wash, dry scratch strokes, wet blooming splashes, pale blank negative space.</i>"),
    img_path_input,
    style_preservation_slider,
    negative_space_slider,
    ink_depth_slider,
    brush_slider,
    btn_generate,
    output_area
])

display(ui)

A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "D:\Anaconda\envs\AD\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'


Loading ControlNet edge-locking controller...
Loading SD 1.5 Img2Img ControlNet pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet_img2img.StableDiffusionControlNetImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


[Success] Adaptive high-contrast control pipeline loaded completely!
